In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)
    if depth <= 3:
        print(root, "->", len(files), "files")

In [ ]:
from pathlib import Path

input_dir = Path("/kaggle/input")

print("Available dataset folders:")
for path in input_dir.iterdir():
    print(path)

# Find the directory containing all six classes
classes = {"cardboard", "glass", "metal", "paper", "plastic", "trash"}

dataset_dir = None

for path in input_dir.rglob("*"):
    if path.is_dir():
        subdirs = {p.name.lower() for p in path.iterdir() if p.is_dir()}
        if classes.issubset(subdirs):
            dataset_dir = path
            break

if dataset_dir is None:
    raise FileNotFoundError(
        "Could not find all six class folders. "
        "Check the dataset structure printed above."
    )

print("\nDataset found at:", dataset_dir)

In [ ]:
from pathlib import Path

extensions = {".jpg", ".jpeg", ".png", ".webp"}

class_counts = {}

for class_name in sorted(
    ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
):
    folder = dataset_dir / class_name

    images = [
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in extensions
    ]

    class_counts[class_name] = len(images)
    print(f"{class_name}: {len(images)} images")

print("\nTotal images:", sum(class_counts.values()))

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

SEED = 42

classes = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
extensions = {".jpg", ".jpeg", ".png", ".webp"}

# Build the image manifest
records = []

for class_name in classes:
    folder = dataset_dir / class_name

    for image_path in sorted(folder.iterdir()):
        if image_path.is_file() and image_path.suffix.lower() in extensions:
            records.append({
                "sample_id": f"{class_name}_{image_path.stem}",
                "path": str(image_path),
                "label": class_name
            })

df = pd.DataFrame(records)

assert len(df) == 2527, f"Expected 2527 images, found {len(df)}"
assert not df["path"].duplicated().any(), "Duplicate paths detected"

# First: 50% training, 50% temporary pool
train_df, remaining = train_test_split(
    df,
    test_size=0.50,
    stratify=df["label"],
    random_state=SEED
)

# From the remaining 50%:
# 20% of total = holdout, 20% = edge stream, 10% = validation
val_df, remaining = train_test_split(
    remaining,
    test_size=0.80,
    stratify=remaining["label"],
    random_state=SEED
)

holdout_df, edge_df = train_test_split(
    remaining,
    test_size=0.50,
    stratify=remaining["label"],
    random_state=SEED
)

train_df = train_df.copy()
val_df = val_df.copy()
holdout_df = holdout_df.copy()
edge_df = edge_df.copy()

train_df["split"] = "train"
val_df["split"] = "validation"
holdout_df["split"] = "clean_holdout"
edge_df["split"] = "edge_stream"

manifest = pd.concat(
    [train_df, val_df, holdout_df, edge_df],
    ignore_index=True
)

# Save the reproducible manifest
output_dir = Path("/kaggle/working/edgeloop")
output_dir.mkdir(parents=True, exist_ok=True)

manifest_path = output_dir / "dataset_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("Split sizes:")
print(manifest["split"].value_counts())

print("\nClass distribution by split:")
print(pd.crosstab(manifest["split"], manifest["label"]))

print("\nSaved manifest to:", manifest_path)

In [ ]:
# Each image must appear exactly once
assert manifest["path"].is_unique
assert len(manifest) == 2527

# Ensure all four splits exist
assert set(manifest["split"]) == {
    "train", "validation", "clean_holdout", "edge_stream"
}

# Ensure every class is represented in each split
counts = pd.crosstab(manifest["split"], manifest["label"])
assert (counts > 0).all().all()

print("SUCCESS: Split verification passed.")
print("No image path appears in more than one split.")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = torch.device("cuda")
else:
    print("Using CPU")
    device = torch.device("cpu")

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, f1_score, classification_report
from pathlib import Path

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

manifest = pd.read_csv("/kaggle/working/edgeloop/dataset_manifest.csv")

classes = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
class_to_idx = {name: i for i, name in enumerate(classes)}

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class TrashDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        with Image.open(row["path"]) as image:
            image = image.convert("RGB")
            image = self.transform(image)

        label = class_to_idx[row["label"]]
        return image, label

train_data = TrashDataset(
    manifest[manifest["split"] == "train"],
    train_transform
)

val_data = TrashDataset(
    manifest[manifest["split"] == "validation"],
    eval_transform
)

train_loader = DataLoader(
    train_data, batch_size=32, shuffle=True, num_workers=2
)

val_loader = DataLoader(
    val_data, batch_size=32, shuffle=False, num_workers=2
)

weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights)

# Replace the final layer for six waste classes
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features, len(classes)
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(), lr=0.0003, weight_decay=0.0001
)

print("Training images:", len(train_data))
print("Validation images:", len(val_data))
print("Classes:", classes)
print("Model: MobileNetV2")

In [ ]:
import copy

EPOCHS = 10
best_val_f1 = -1.0
best_state = None

history = []

for epoch in range(EPOCHS):
    model.train()
    train_loss_sum = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * images.size(0)

    model.eval()
    val_true = []
    val_pred = []
    val_loss_sum = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss_sum += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            val_true.extend(labels.cpu().tolist())
            val_pred.extend(predictions.cpu().tolist())

    train_loss = train_loss_sum / len(train_data)
    val_loss = val_loss_sum / len(val_data)
    val_acc = accuracy_score(val_true, val_pred)
    val_f1 = f1_score(
        val_true, val_pred, average="macro", zero_division=0
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1
    })

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Val accuracy: {val_acc:.4f} | "
        f"Val macro-F1: {val_f1:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = copy.deepcopy(model.state_dict())

# Restore best validation checkpoint
model.load_state_dict(best_state)

output_dir = Path("/kaggle/working/edgeloop")
output_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = output_dir / "mobilenetv2_baseline.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "classes": classes,
    "class_to_idx": class_to_idx,
    "seed": SEED,
    "best_val_macro_f1": best_val_f1,
    "input_size": 224,
    "normalization_mean": [0.485, 0.456, 0.406],
    "normalization_std": [0.229, 0.224, 0.225]
}, checkpoint_path)

pd.DataFrame(history).to_csv(
    output_dir / "training_history.csv", index=False
)

print("\nBest validation macro-F1:", round(best_val_f1, 4))
print("Saved checkpoint:", checkpoint_path)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import json

# Use the existing trained model and evaluation transform
holdout_df = manifest[
    manifest["split"] == "clean_holdout"
].copy()

holdout_data = TrashDataset(holdout_df, eval_transform)

holdout_loader = DataLoader(
    holdout_data,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in holdout_loader:
        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        y_true.extend(labels.tolist())
        y_pred.extend(predictions.cpu().tolist())

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(
    y_true, y_pred, average="macro", zero_division=0
)

report = classification_report(
    y_true,
    y_pred,
    labels=list(range(len(classes))),
    target_names=classes,
    output_dict=True,
    zero_division=0
)

print(f"Clean holdout accuracy: {accuracy:.4f}")
print(f"Clean holdout macro-F1: {macro_f1:.4f}")
print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(classes))),
    target_names=classes,
    zero_division=0
))

print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))

# Save evaluation artifacts
results_dir = Path("/kaggle/working/edgeloop")
results_dir.mkdir(parents=True, exist_ok=True)

with open(results_dir / "baseline_holdout_metrics.json", "w") as f:
    json.dump({
        "model": "MobileNetV2",
        "seed": SEED,
        "holdout_images": len(holdout_data),
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "classification_report": report
    }, f, indent=2)

print("\nMetrics saved to baseline_holdout_metrics.json")

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/edgeloop_baseline_artifacts",
    "zip",
    "/kaggle/working/edgeloop"
)

print("Created /kaggle/working/edgeloop_baseline_artifacts.zip")

In [ ]:
from pathlib import Path
from PIL import Image, ImageFilter, ImageEnhance
import pandas as pd
import numpy as np

manifest = pd.read_csv(
    "/kaggle/working/edgeloop/dataset_manifest.csv"
)

edge_df = manifest[
    manifest["split"] == "edge_stream"
].copy()

corruption_root = Path("/kaggle/working/edgeloop/corrupted_edge")
corruption_root.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
records = []

for _, row in edge_df.iterrows():
    image_path = Path(row["path"])

    with Image.open(image_path) as im:
        image = im.convert("RGB")

    # Preserve the clean edge-stream sample
    records.append({
        "sample_id": row["sample_id"],
        "path": str(image_path),
        "label": row["label"],
        "condition": "clean"
    })

    # Blur
    blur = image.filter(ImageFilter.GaussianBlur(radius=2.0))

    # Low light
    low_light = ImageEnhance.Brightness(image).enhance(0.35)

    # Gaussian-like pixel noise
    arr = np.asarray(image).astype(np.float32)
    noise = rng.normal(0, 20, arr.shape)
    noisy_arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    noisy = Image.fromarray(noisy_arr)

    variants = {
        "blur": blur,
        "low_light": low_light,
        "noise": noisy
    }

    for condition, variant in variants.items():
        output_path = (
            corruption_root
            / condition
            / f"{row['sample_id']}.jpg"
        )

        output_path.parent.mkdir(parents=True, exist_ok=True)
        variant.save(output_path, quality=95)

        records.append({
            "sample_id": row["sample_id"],
            "path": str(output_path),
            "label": row["label"],
            "condition": condition
        })

corruption_manifest = pd.DataFrame(records)

corruption_manifest.to_csv(
    "/kaggle/working/edgeloop/corruption_manifest.csv",
    index=False
)

print("Edge-stream source images:", len(edge_df))
print("Total condition-image records:", len(corruption_manifest))
print("\nRecords per condition:")
print(corruption_manifest["condition"].value_counts())
print("\nSaved corruption manifest.")

In [ ]:
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score

corruption_manifest = pd.read_csv(
    "/kaggle/working/edgeloop/corruption_manifest.csv"
)

class CorruptionDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform
        self.class_to_idx = {
            name: i for i, name in enumerate(classes)
        }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        with Image.open(row["path"]) as image:
            image = self.transform(image.convert("RGB"))

        label = self.class_to_idx[row["label"]]
        return image, label

model.eval()
results = []

for condition in ["clean", "blur", "low_light", "noise"]:
    subset = corruption_manifest[
        corruption_manifest["condition"] == condition
    ].copy()

    dataset = CorruptionDataset(subset, eval_transform)
    loader = DataLoader(
        dataset, batch_size=32, shuffle=False, num_workers=2
    )

    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            y_true.extend(labels.tolist())
            y_pred.extend(predictions.cpu().tolist())

    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(
        y_true, y_pred, average="macro", zero_division=0
    )

    results.append({
        "condition": condition,
        "images": len(dataset),
        "accuracy": accuracy,
        "macro_f1": macro_f1
    })

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

output_path = Path("/kaggle/working/edgeloop")
results_df.to_csv(
    output_path / "corruption_evaluation.csv", index=False
)

print("\nSaved:", output_path / "corruption_evaluation.csv")

In [ ]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import DataLoader

manifest = pd.read_csv(
    "/kaggle/working/edgeloop/dataset_manifest.csv"
)

edge_df = manifest[
    manifest["split"] == "edge_stream"
].copy()

edge_df["condition"] = "clean"
corruption_manifest = pd.read_csv(
    "/kaggle/working/edgeloop/corruption_manifest.csv"
)

# Use the simulated incoming stream, including corrupted images.
stream_df = corruption_manifest[
    corruption_manifest["condition"].isin(
        ["clean", "blur", "low_light", "noise"]
    )
].copy()

class StreamDataset(CorruptionDataset):
    pass

stream_data = StreamDataset(stream_df, eval_transform)
stream_loader = DataLoader(
    stream_data, batch_size=32, shuffle=False, num_workers=2
)

model.eval()
records = []

with torch.no_grad():
    offset = 0

    for images, labels in stream_loader:
        images = images.to(device)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)

        confidence, predictions = probs.max(dim=1)
        entropy = -(
            probs * torch.log(probs.clamp_min(1e-8))
        ).sum(dim=1)

        batch_size = len(images)

        for i in range(batch_size):
            row = stream_df.iloc[offset + i]

            records.append({
                "sample_id": row["sample_id"],
                "path": row["path"],
                "condition": row["condition"],
                "predicted_class": classes[predictions[i].item()],
                "confidence": confidence[i].item(),
                "entropy": entropy[i].item()
            })

        offset += batch_size

predictions_df = pd.DataFrame(records)

out_path = Path("/kaggle/working/edgeloop")
predictions_df.to_csv(
    out_path / "edge_predictions.csv", index=False
)

print("Prediction records:", len(predictions_df))
print("\nMean confidence by condition:")
print(predictions_df.groupby("condition")["confidence"].mean())

print("\nMean entropy by condition:")
print(predictions_df.groupby("condition")["entropy"].mean())

print("\nSaved edge_predictions.csv")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

SEED = 42
UPLOAD_FRACTION = 0.10

predictions_df = pd.read_csv(
    "/kaggle/working/edgeloop/edge_predictions.csv"
)

n = len(predictions_df)
budget = int(np.ceil(n * UPLOAD_FRACTION))

rng = np.random.default_rng(SEED)

# Strategy 1: Random sampling
random_indices = rng.choice(
    predictions_df.index,
    size=budget,
    replace=False
)

random_selected = set(random_indices)

# Strategy 2: Uncertainty sampling
uncertain_indices = (
    predictions_df["entropy"]
    .nlargest(budget)
    .index
)

uncertain_selected = set(uncertain_indices)

# Record the selected status for each strategy
predictions_df["random_selected"] = (
    predictions_df.index.isin(random_selected)
)

predictions_df["uncertainty_selected"] = (
    predictions_df.index.isin(uncertain_selected)
)

predictions_df["upload_reason"] = "not_selected"

predictions_df.loc[
    predictions_df["random_selected"],
    "upload_reason"
] = "random"

predictions_df.loc[
    predictions_df["uncertainty_selected"],
    "upload_reason"
] = "uncertainty"

# If an image was selected by both strategies, retain both indicators.
output_path = Path("/kaggle/working/edgeloop")
predictions_df.to_csv(
    output_path / "selection_results.csv",
    index=False
)

print("Total stream records:", n)
print("Upload budget:", budget)
print(
    "Random selected:",
    int(predictions_df["random_selected"].sum())
)
print(
    "Uncertainty selected:",
    int(predictions_df["uncertainty_selected"].sum())
)

print("\nUncertainty selection by condition:")
print(
    predictions_df.groupby("condition")[
        "uncertainty_selected"
    ].agg(["sum", "mean"])
)

print("\nSaved selection_results.csv")

In [ ]:
import pandas as pd
from pathlib import Path

base = Path("/kaggle/working/edgeloop")

selected = pd.read_csv(base / "selection_results.csv")
truth = pd.read_csv(base / "corruption_manifest.csv")

# Match labels by the unique record identity:
# sample_id + condition
evaluation = selected.merge(
    truth[["sample_id", "condition", "label"]],
    on=["sample_id", "condition"],
    how="left",
    validate="one_to_one"
)

assert evaluation["label"].notna().all()

evaluation["is_error"] = (
    evaluation["predicted_class"] != evaluation["label"]
)

for strategy in ["random_selected", "uncertainty_selected"]:
    subset = evaluation[evaluation[strategy]]

    print(f"\n{strategy}")
    print("Selected records:", len(subset))
    print("Errors:", int(subset["is_error"].sum()))
    print("Error rate:", round(subset["is_error"].mean(), 4))
    print("\nErrors by condition:")
    print(
        subset.groupby("condition")["is_error"]
        .agg(["sum", "mean"])
    )

print("\nOverall stream error rate:",
      round(evaluation["is_error"].mean(), 4))

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageFilter
from scipy.stats import ks_2samp

base = Path("/kaggle/working/edgeloop")

stream = pd.read_csv(base / "corruption_manifest.csv")

def extract_features(path):
    with Image.open(path) as image:
        gray = image.convert("L").resize((128, 128))

        pixels = np.asarray(gray, dtype=np.float32)

        # Normalized average brightness
        brightness = pixels.mean() / 255.0

        # Normalized contrast proxy
        contrast = pixels.std() / 255.0

        # Edge-strength proxy
        edges = np.asarray(
            gray.filter(ImageFilter.FIND_EDGES),
            dtype=np.float32
        )
        edge_strength = edges.std() / 255.0

    return brightness, contrast, edge_strength

features = stream["path"].apply(extract_features)

stream[["brightness", "contrast", "edge_strength"]] = pd.DataFrame(
    features.tolist(),
    index=stream.index
)

stream.to_csv(base / "stream_features.csv", index=False)

print("Extracted image features:", len(stream))
print(
    stream.groupby("condition")[
        ["brightness", "contrast", "edge_strength"]
    ].mean().round(4)
)

In [ ]:
feature_names = ["brightness", "contrast", "edge_strength"]

# The detector uses features only; condition is used only to organize
# this retrospective experiment's evaluation groups.
reference = stream.loc[
    stream["condition"] == "clean", feature_names
]

results = []

for condition in ["clean", "blur", "low_light", "noise"]:
    incoming = stream.loc[
        stream["condition"] == condition, feature_names
    ]

    for feature in feature_names:
        statistic, p_value = ks_2samp(
            reference[feature],
            incoming[feature]
        )

        results.append({
            "condition_for_evaluation": condition,
            "feature": feature,
            "ks_statistic": statistic,
            "p_value": p_value,
            "flagged_at_0.01": p_value < 0.01
        })

drift_results = pd.DataFrame(results)

print(drift_results.to_string(index=False))

drift_results.to_csv(
    base / "drift_detection_results.csv",
    index=False
)

print("\nSaved drift_detection_results.csv")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageFilter
from scipy.stats import ks_2samp

base = Path("/kaggle/working/edgeloop")
stream = pd.read_csv(base / "corruption_manifest.csv")

def extract_noise_features(path):
    with Image.open(path) as image:
        gray = image.convert("L").resize((128, 128))
        pixels = np.asarray(gray, dtype=np.float32)

    # Approximate high-frequency residual:
    # original grayscale image minus a smoothed version.
    smooth = np.asarray(
        Image.fromarray(pixels.astype(np.uint8)).filter(
            ImageFilter.GaussianBlur(radius=1.5)
        ),
        dtype=np.float32
    )

    residual = pixels - smooth

    return {
        "high_frequency_std": float(residual.std()),
        "high_frequency_mean_abs": float(np.abs(residual).mean())
    }

new_features = stream["path"].apply(extract_noise_features)

stream["high_frequency_std"] = [
    item["high_frequency_std"] for item in new_features
]
stream["high_frequency_mean_abs"] = [
    item["high_frequency_mean_abs"] for item in new_features
]

feature_cols = [
    "brightness",
    "contrast",
    "edge_strength",
    "high_frequency_std",
    "high_frequency_mean_abs"
]

# Recalculate existing features if this cell is run independently.
def base_features(path):
    with Image.open(path) as image:
        gray = image.convert("L").resize((128, 128))
        pixels = np.asarray(gray, dtype=np.float32)

        edges = np.asarray(
            gray.filter(ImageFilter.FIND_EDGES),
            dtype=np.float32
        )

    return {
        "brightness": pixels.mean() / 255.0,
        "contrast": pixels.std() / 255.0,
        "edge_strength": edges.std() / 255.0
    }

base_values = stream["path"].apply(base_features)

for col in ["brightness", "contrast", "edge_strength"]:
    stream[col] = [item[col] for item in base_values]

reference = stream.loc[
    stream["condition"] == "clean", feature_cols
]

results = []

for condition in ["clean", "blur", "low_light", "noise"]:
    incoming = stream.loc[
        stream["condition"] == condition, feature_cols
    ]

    for feature in feature_cols:
        statistic, p_value = ks_2samp(
            reference[feature], incoming[feature]
        )

        results.append({
            "condition": condition,
            "feature": feature,
            "ks_statistic": statistic,
            "p_value": p_value,
            "flagged_at_0.01": p_value < 0.01
        })

drift_results = pd.DataFrame(results)

print(drift_results.to_string(index=False))

stream.to_csv(base / "stream_features.csv", index=False)
drift_results.to_csv(base / "drift_detection_results_v2.csv", index=False)

print("\nSaved updated feature and drift results.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

base = Path("/kaggle/working/edgeloop")

predictions = pd.read_csv(base / "edge_predictions.csv")
features = pd.read_csv(base / "stream_features.csv")

df = predictions.merge(
    features[
        ["sample_id", "condition", "brightness",
         "high_frequency_mean_abs"]
    ],
    on=["sample_id", "condition"],
    how="left",
    validate="one_to_one"
)

# Provisional thresholds based on the clean-reference feature values.
# Replace these with thresholds calibrated on validation windows.
brightness_drift = df["brightness"] < 0.40
noise_drift = df["high_frequency_mean_abs"] > 0.10

df["drift_flag"] = brightness_drift | noise_drift

# Same record-level budget as the previous experiment
budget = int(np.ceil(len(df) * 0.10))

# Drift-aware strategy: prioritize uncertain records when drift is flagged.
# If fewer than budget are flagged, fill remaining slots with highest entropy.
flagged = df[df["drift_flag"]].sort_values(
    "entropy", ascending=False
)

unflagged = df[~df["drift_flag"]].sort_values(
    "entropy", ascending=False
)

selected = pd.concat(
    [flagged, unflagged],
    ignore_index=True
).head(budget)

selected_ids = set(selected.index)
df["drift_aware_selected"] = df.index.isin(selected_ids)

df.to_csv(base / "drift_aware_selection.csv", index=False)

print("Total records:", len(df))
print("Upload budget:", budget)
print("Drift-flagged records:", int(df["drift_flag"].sum()))
print("Selected records:", int(df["drift_aware_selected"].sum()))

print("\nSelected records by condition (evaluation only):")
print(
    df[df["drift_aware_selected"]]
    .groupby("condition").size()
)

print("\nSaved drift_aware_selection.csv")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

base = Path("/kaggle/working/edgeloop")
features = pd.read_csv(base / "stream_features.csv")

# Use only clean reference images to calibrate thresholds.
reference = features[features["condition"] == "clean"]

# Set thresholds from the clean reference distribution.
brightness_low = reference["brightness"].quantile(0.01)
brightness_high = reference["brightness"].quantile(0.99)

noise_high = reference["high_frequency_mean_abs"].quantile(0.99)

print("Calibrated thresholds:")
print("Brightness lower:", round(brightness_low, 4))
print("Brightness upper:", round(brightness_high, 4))
print("High-frequency upper:", round(noise_high, 4))

features["drift_flag"] = (
    (features["brightness"] < brightness_low)
    | (features["brightness"] > brightness_high)
    | (
        features["high_frequency_mean_abs"] > noise_high
    )
)

print("\nFlagged fraction by condition (offline evaluation):")
print(
    features.groupby("condition")["drift_flag"]
    .mean()
    .round(3)
)

features.to_csv(base / "stream_features_calibrated.csv", index=False)
print("\nSaved stream_features_calibrated.csv")

In [ ]:
import pandas as pd
from pathlib import Path

base = Path("/kaggle/working/edgeloop")
features = pd.read_csv(base / "stream_features.csv")

cols = [
    "brightness",
    "contrast",
    "edge_strength",
    "high_frequency_std",
    "high_frequency_mean_abs"
]

summary = features.groupby("condition")[cols].agg(
    ["mean", "median", "std"]
)

print(summary.round(4).to_string())

In [ ]:
import pandas as pd
from pathlib import Path

base = Path("/kaggle/working/edgeloop")
features = pd.read_csv(base / "stream_features.csv")

# Use clean images only to establish reference thresholds
clean = features[features["condition"] == "clean"]

# Reference thresholds: 1st and 99th percentiles
brightness_low = clean["brightness"].quantile(0.01)
brightness_high = clean["brightness"].quantile(0.99)
edge_low = clean["edge_strength"].quantile(0.01)
hf_low = clean["high_frequency_mean_abs"].quantile(0.01)
hf_high = clean["high_frequency_mean_abs"].quantile(0.99)

print("Calibrated clean-reference thresholds")
print(f"Brightness: {brightness_low:.4f} to {brightness_high:.4f}")
print(f"Edge strength minimum: {edge_low:.4f}")
print(f"High-frequency mean absolute minimum: {hf_low:.4f}")
print(f"High-frequency mean absolute maximum: {hf_high:.4f}")

# Flag an image if any feature falls outside its reference range
features["drift_flag"] = (
    (features["brightness"] < brightness_low) |
    (features["brightness"] > brightness_high) |
    (features["edge_strength"] < edge_low) |
    (features["high_frequency_mean_abs"] < hf_low) |
    (features["high_frequency_mean_abs"] > hf_high)
)

print("\nFlagged fraction by condition")
print(
    features.groupby("condition")["drift_flag"]
    .mean()
    .round(3)
    .to_string()
)

features.to_csv(base / "stream_features_with_drift.csv", index=False)
print("\nSaved: stream_features_with_drift.csv")

In [6]:
from pathlib import Path
import zipfile

zip_path = Path("/kaggle/input/YOUR-DATASET-NAME/EdgeLoop_Kaggle_Research_v0.3.zip")
extract_dir = Path("/kaggle/working")

if not zip_path.exists():
    print("ZIP not found. Check the attached dataset path.")
else:
    with zipfile.ZipFile(zip_path, "r") as z:
        print("Archive files:", len(z.namelist()))
        z.extractall(extract_dir)

    print("Restoration complete.")
    print("Project folder:", extract_dir / "edgeloop")
    print("Folder exists:", (extract_dir / "edgeloop").exists())

ZIP not found. Check the attached dataset path.


In [9]:
from pathlib import Path
import shutil

# Locate the attached EdgeLoop project folder
root = Path("/kaggle/input")
matches = [
    p for p in root.rglob("mobilenetv2_baseline.pth")
    if p.is_file()
]

if not matches:
    print("Model checkpoint not found. We need to inspect the attached input path.")
else:
    source_dir = matches[0].parent
    target_dir = Path("/kaggle/working/edgeloop")

    target_dir.mkdir(parents=True, exist_ok=True)

    for item in source_dir.iterdir():
        destination = target_dir / item.name

        if item.is_dir():
            if not destination.exists():
                shutil.copytree(item, destination)
        elif not destination.exists():
            shutil.copy2(item, destination)

    print("Restored from:", source_dir)
    print("Working folder:", target_dir)
    print("\nRestored files:")
    for item in sorted(target_dir.iterdir()):
        print(item.name)

Restored from: /kaggle/input/datasets/balajimali/project-file/edgeloop
Working folder: /kaggle/working/edgeloop

Restored files:
baseline_holdout_metrics.json
checkpoint_v0.3.json
corrupted_edge
corruption_evaluation.csv
corruption_manifest.csv
dataset_manifest.csv
drift_aware_selection.csv
drift_detection_results.csv
drift_detection_results_v2.csv
edge_predictions.csv
mobilenetv2_baseline.pth
selection_results.csv
stream_features.csv
stream_features_calibrated.csv
stream_features_with_drift.csv
training_history.csv


In [10]:
from pathlib import Path
import json
import pandas as pd

base = Path("/kaggle/working/edgeloop")

# Verify key files
required = [
    "mobilenetv2_baseline.pth",
    "dataset_manifest.csv",
    "baseline_holdout_metrics.json",
    "corruption_evaluation.csv",
    "edge_predictions.csv",
    "selection_results.csv",
    "stream_features_with_drift.csv",
    "checkpoint_v0.3.json",
]

print("FILE VERIFICATION")
for name in required:
    path = base / name
    print(f"{name}: {'OK' if path.is_file() else 'MISSING'}")

# Read saved baseline metrics
with open(base / "baseline_holdout_metrics.json") as f:
    metrics = json.load(f)

print("\nBASELINE HOLDOUT METRICS")
print(json.dumps(metrics, indent=2))

# Check experiment data sizes
for name in [
    "corruption_evaluation.csv",
    "edge_predictions.csv",
    "selection_results.csv",
    "stream_features_with_drift.csv",
]:
    path = base / name
    if path.is_file():
        print(f"{name}: {len(pd.read_csv(path))} rows")

FILE VERIFICATION
mobilenetv2_baseline.pth: OK
dataset_manifest.csv: OK
baseline_holdout_metrics.json: OK
corruption_evaluation.csv: OK
edge_predictions.csv: OK
selection_results.csv: OK
stream_features_with_drift.csv: OK
checkpoint_v0.3.json: OK

BASELINE HOLDOUT METRICS
{
  "model": "MobileNetV2",
  "seed": 42,
  "holdout_images": 506,
  "accuracy": 0.9130434782608695,
  "macro_f1": 0.8943403100172601,
  "classification_report": {
    "cardboard": {
      "precision": 0.9871794871794872,
      "recall": 0.9506172839506173,
      "f1-score": 0.9685534591194969,
      "support": 81.0
    },
    "glass": {
      "precision": 0.8888888888888888,
      "recall": 0.88,
      "f1-score": 0.8844221105527639,
      "support": 100.0
    },
    "metal": {
      "precision": 0.9012345679012346,
      "recall": 0.8902439024390244,
      "f1-score": 0.8957055214723927,
      "support": 82.0
    },
    "paper": {
      "precision": 0.9349593495934959,
      "recall": 0.9663865546218487,
      "f1-s

In [11]:
from pathlib import Path
import pandas as pd

base = Path("/kaggle/working/edgeloop")

selection = pd.read_csv(base / "selection_results.csv")
predictions = pd.read_csv(base / "edge_predictions.csv")

print("SELECTION COLUMNS")
print(selection.columns.tolist())

print("\nSELECTION SAMPLE")
print(selection.head(5).to_string(index=False))

print("\nPREDICTION COLUMNS")
print(predictions.columns.tolist())

print("\nPREDICTION SAMPLE")
print(predictions.head(5).to_string(index=False))

SELECTION COLUMNS
['sample_id', 'path', 'condition', 'predicted_class', 'confidence', 'entropy', 'random_selected', 'uncertainty_selected', 'upload_reason']

SELECTION SAMPLE
            sample_id                                                                                   path condition predicted_class  confidence  entropy  random_selected  uncertainty_selected upload_reason
       glass_glass320        /kaggle/input/datasets/balajimali/ml-project/dataset-resized/glass/glass320.jpg     clean           glass    0.998179 0.015024            False                 False  not_selected
       glass_glass320                        /kaggle/working/edgeloop/corrupted_edge/blur/glass_glass320.jpg      blur           glass    0.998061 0.015085            False                 False  not_selected
       glass_glass320                   /kaggle/working/edgeloop/corrupted_edge/low_light/glass_glass320.jpg low_light           glass    0.999240 0.007107            False                 False  no

In [12]:
from pathlib import Path
import pandas as pd
import numpy as np

base = Path("/kaggle/working/edgeloop")

selection = pd.read_csv(base / "selection_results.csv")
manifest = pd.read_csv(base / "corruption_manifest.csv")

# Join true labels for offline evaluation only
data = selection.merge(
    manifest[["sample_id", "condition", "label"]],
    on=["sample_id", "condition"],
    how="left",
    validate="one_to_one"
)

data["is_error"] = (
    data["predicted_class"].astype(str).str.lower()
    != data["label"].astype(str).str.lower()
)

print("Missing true labels:", data["label"].isna().sum())
print("Total records:", len(data))
print("Overall error rate:", round(data["is_error"].mean(), 4))

for strategy in ["random_selected", "uncertainty_selected"]:
    selected = data[data[strategy].astype(bool)]

    print(f"\n--- {strategy} ---")
    print("Selected records:", len(selected))
    print("Selected error rate:", round(selected["is_error"].mean(), 4))
    print("Errors captured:", int(selected["is_error"].sum()))
    print("Share of all errors captured:",
          round(selected["is_error"].sum() / data["is_error"].sum(), 4)
          if data["is_error"].sum() else 0)

    print("\nBy condition:")
    print(
        selected.groupby("condition")["is_error"]
        .agg(["count", "mean", "sum"])
        .round(4)
        .to_string()
    )

Missing true labels: 0
Total records: 2024
Overall error rate: 0.1823

--- random_selected ---
Selected records: 203
Selected error rate: 0.2266
Errors captured: 46
Share of all errors captured: 0.1247

By condition:
           count    mean  sum
condition                    
blur          44  0.1591    7
clean         55  0.1455    8
low_light     64  0.2969   19
noise         40  0.3000   12

--- uncertainty_selected ---
Selected records: 203
Selected error rate: 0.5911
Errors captured: 120
Share of all errors captured: 0.3252

By condition:
           count    mean  sum
condition                    
blur          42  0.5238   22
clean         12  0.9167   11
low_light     54  0.5741   31
noise         95  0.5895   56


In [14]:
clean_selected = data[
    (data["condition"] == "clean") &
    (data["uncertainty_selected"].astype(bool))
].copy()

print("Clean records selected:", len(clean_selected))
print("Clean records misclassified:", int(clean_selected["is_error"].sum()))

print(
    clean_selected[
        ["sample_id", "label", "predicted_class", "confidence", "entropy", "is_error"]
    ]
    .sort_values("entropy", ascending=False)
    .to_string(index=False)
)

Clean records selected: 12
Clean records misclassified: 11
             sample_id     label predicted_class  confidence  entropy  is_error
        metal_metal176     metal           paper    0.337409 1.560163      True
         trash_trash86     trash           paper    0.358025 1.421576      True
cardboard_cardboard380 cardboard       cardboard    0.497890 1.395828     False
         trash_trash93     trash           glass    0.388024 1.385555      True
    plastic_plastic456   plastic           metal    0.359443 1.367393      True
        metal_metal166     metal           glass    0.454792 1.303850      True
         metal_metal14     metal           trash    0.339501 1.175211      True
cardboard_cardboard290 cardboard         plastic    0.489588 1.151872      True
cardboard_cardboard286 cardboard           paper    0.522183 1.141584      True
        metal_metal165     metal         plastic    0.577570 1.124402      True
    plastic_plastic303   plastic           glass    0.607367 

In [15]:
# Compare unique source images selected by each strategy
for strategy in ["random_selected", "uncertainty_selected"]:
    selected = data[data[strategy].astype(bool)]

    print(f"\n{strategy}")
    print("Selected stream records:", len(selected))
    print("Unique source images:", selected["sample_id"].nunique())
    print("Unique images with at least one error:",
          selected.loc[selected["is_error"], "sample_id"].nunique())

# Check whether selected variants from the same image are repeated
uncertain = data[data["uncertainty_selected"].astype(bool)]
print("\nUncertainty-selected records per condition:")
print(uncertain["condition"].value_counts().to_string())

print("\nSource images with multiple uncertainty-selected variants:")
print(
    uncertain.groupby("sample_id")["condition"]
    .count()
    .value_counts()
    .sort_index()
    .to_string()
)


random_selected
Selected stream records: 203
Unique source images: 173
Unique images with at least one error: 43

uncertainty_selected
Selected stream records: 203
Unique source images: 134
Unique images with at least one error: 89

Uncertainty-selected records per condition:
condition
noise        95
low_light    54
blur         42
clean        12

Source images with multiple uncertainty-selected variants:
condition
1    80
2    40
3    13
4     1


In [16]:
from pathlib import Path
import pandas as pd

base = Path("/kaggle/working/edgeloop")
data = pd.read_csv(base / "selection_results.csv")
manifest = pd.read_csv(base / "corruption_manifest.csv")

data = data.merge(
    manifest[["sample_id", "condition", "label"]],
    on=["sample_id", "condition"],
    how="left",
    validate="one_to_one"
)

data["is_error"] = (
    data["predicted_class"].astype(str).str.lower()
    != data["label"].astype(str).str.lower()
)

budget = int(data["uncertainty_selected"].sum())

# Rank highest-entropy variant first for each source image
ranked = data.sort_values("entropy", ascending=False).copy()
unique_first = ranked.drop_duplicates("sample_id", keep="first")

# First choose one record per source image, then fill remaining budget
chosen = unique_first.head(budget)
if len(chosen) < budget:
    remaining = ranked[~ranked.index.isin(chosen.index)]
    chosen = pd.concat([chosen, remaining.head(budget - len(chosen))])

data["unique_aware_selected"] = data.index.isin(chosen.index)

print("Budget:", budget)
print("Selected records:", int(data["unique_aware_selected"].sum()))
print(
    "Unique source images:",
    data.loc[data["unique_aware_selected"], "sample_id"].nunique()
)
print(
    "Errors captured:",
    int(data.loc[data["unique_aware_selected"], "is_error"].sum())
)
print(
    "Unique error-containing images:",
    data.loc[
        data["unique_aware_selected"] & data["is_error"], "sample_id"
    ].nunique()
)

data.drop(columns=["label", "is_error"]).to_csv(
    base / "selection_results_v2.csv", index=False
)
print("Saved selection_results_v2.csv")

Budget: 203
Selected records: 203
Unique source images: 203
Errors captured: 102
Unique error-containing images: 102
Saved selection_results_v2.csv


In [17]:
from pathlib import Path
import shutil
import json

base = Path("/kaggle/working/edgeloop")

# Update checkpoint metadata
checkpoint_path = base / "checkpoint_v0.3.json"

with open(checkpoint_path) as f:
    checkpoint = json.load(f)

checkpoint["latest_experiment"] = {
    "name": "Unique-image-aware uncertainty selection",
    "budget_records": 203,
    "unique_source_images_selected": 203,
    "errors_captured": 102,
    "output_file": "selection_results_v2.csv"
}

with open(checkpoint_path, "w") as f:
    json.dump(checkpoint, f, indent=2)

# Regenerate archive with all current experiment artifacts
archive = shutil.make_archive(
    "/kaggle/working/EdgeLoop_Kaggle_Research_v0.4",
    "zip",
    root_dir="/kaggle/working",
    base_dir="edgeloop"
)

print("Saved checkpoint:", archive)
print("Archive size (MB):", round(Path(archive).stat().st_size / (1024 * 1024), 2))
print("Selection results v2 exists:",
      (base / "selection_results_v2.csv").exists())

Saved checkpoint: /kaggle/working/EdgeLoop_Kaggle_Research_v0.4.zip
Archive size (MB): 99.32
Selection results v2 exists: True


In [18]:
from pathlib import Path
import pandas as pd
import numpy as np

base = Path("/kaggle/working/edgeloop")

# Load predictions, drift flags, and ground-truth labels
predictions = pd.read_csv(base / "edge_predictions.csv")
features = pd.read_csv(base / "stream_features_with_drift.csv")
manifest = pd.read_csv(base / "corruption_manifest.csv")

data = predictions.merge(
    features[["sample_id", "condition", "drift_flag"]],
    on=["sample_id", "condition"],
    how="left",
    validate="one_to_one"
)

data = data.merge(
    manifest[["sample_id", "condition", "label"]],
    on=["sample_id", "condition"],
    how="left",
    validate="one_to_one"
)

data["is_error"] = (
    data["predicted_class"].astype(str).str.lower()
    != data["label"].astype(str).str.lower()
)

budget = 203

# Strategy 1: existing random selection
random_ids = set(
    pd.read_csv(base / "selection_results.csv")
    .loc[lambda df: df["random_selected"].astype(bool), "sample_id"]
)

# Preserve the exact original random-selected records by record identity
original_selection = pd.read_csv(base / "selection_results.csv")
random_records = set(
    zip(
        original_selection.loc[
            original_selection["random_selected"].astype(bool), "sample_id"
        ],
        original_selection.loc[
            original_selection["random_selected"].astype(bool), "condition"
        ]
    )
)
data["random_selected"] = [
    (sid, cond) in random_records
    for sid, cond in zip(data["sample_id"], data["condition"])
]

# Strategy 2: uncertainty, highest entropy first
data["uncertainty_selected"] = False
uncertain_idx = data.nlargest(budget, "entropy").index
data.loc[uncertain_idx, "uncertainty_selected"] = True

# Strategy 3: drift-aware priority
# Rank drift-flagged records first, then use entropy to prioritize within each group
data["drift_aware_selected"] = False
ranked = data.sort_values(
    ["drift_flag", "entropy"],
    ascending=[False, False]
)
drift_idx = ranked.head(budget).index
data.loc[drift_idx, "drift_aware_selected"] = True

# Compare performance
rows = []
for strategy in [
    "random_selected",
    "uncertainty_selected",
    "drift_aware_selected"
]:
    selected = data[data[strategy]]

    rows.append({
        "strategy": strategy,
        "selected_records": len(selected),
        "unique_source_images": selected["sample_id"].nunique(),
        "errors_captured": int(selected["is_error"].sum()),
        "selected_error_rate": round(selected["is_error"].mean(), 4),
        "unique_error_images": selected.loc[
            selected["is_error"], "sample_id"
        ].nunique()
    })

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

comparison.to_csv(base / "selection_strategy_comparison.csv", index=False)
print("\nSaved selection_strategy_comparison.csv")

            strategy  selected_records  unique_source_images  errors_captured  selected_error_rate  unique_error_images
     random_selected               203                   173               46               0.2266                   43
uncertainty_selected               203                   134              120               0.5911                   89
drift_aware_selected               203                   188               83               0.4089                   79

Saved selection_strategy_comparison.csv


In [19]:
selected = data[data["drift_aware_selected"]].copy()

print("Drift-aware selected records by condition:")
print(
    selected.groupby("condition")
    .agg(
        selected_records=("sample_id", "size"),
        error_count=("is_error", "sum"),
        error_rate=("is_error", "mean"),
        unique_images=("sample_id", "nunique")
    )
    .round(4)
    .to_string()
)

print("\nDrift flags among selected records:")
print(selected["drift_flag"].value_counts(dropna=False).to_string())

Drift-aware selected records by condition:
           selected_records  error_count  error_rate  unique_images
condition                                                          
blur                      9            4      0.4444              9
clean                     5            2      0.4000              5
low_light               184           76      0.4130            184
noise                     5            1      0.2000              5

Drift flags among selected records:
drift_flag
True    203


In [21]:
from pathlib import Path
import pandas as pd

base = Path("/kaggle/working/edgeloop")

features = pd.read_csv(base / "stream_features_with_drift.csv")
predictions = pd.read_csv(base / "edge_predictions.csv")
manifest = pd.read_csv(base / "corruption_manifest.csv")

# Keep only the required columns before merging
features = features[
    ["sample_id", "condition", "drift_flag"]
]

predictions = predictions[
    ["sample_id", "condition", "entropy", "predicted_class"]
]

manifest = manifest[
    ["sample_id", "condition", "label"]
]

data = features.merge(
    predictions,
    on=["sample_id", "condition"],
    how="left",
    validate="one_to_one"
)

data = data.merge(
    manifest,
    on=["sample_id", "condition"],
    how="left",
    validate="one_to_one"
)

print("Missing labels:", data["label"].isna().sum())
print("Total records:", len(data))

data["is_error"] = (
    data["predicted_class"].astype(str).str.lower()
    != data["label"].astype(str).str.lower()
)

print("\nDrift flag rate by condition:")
print(
    data.groupby("condition")["drift_flag"]
    .agg(["count", "mean"])
    .round(4)
    .to_string()
)

print("\nError rate by condition and drift flag:")
print(
    data.groupby(["condition", "drift_flag"])["is_error"]
    .agg(["count", "mean"])
    .round(4)
    .to_string()
)

Missing labels: 0
Total records: 2024

Drift flag rate by condition:
           count    mean
condition               
blur         506  0.0652
clean        506  0.0573
low_light    506  1.0000
noise        506  0.0336

Error rate by condition and drift flag:
                      count    mean
condition drift_flag               
blur      False         473  0.1395
          True           33  0.1515
clean     False         477  0.1090
          True           29  0.1034
low_light True          506  0.2312
noise     False         489  0.2536
          True           17  0.1176


In [22]:
from pathlib import Path
import pandas as pd

base = Path("/kaggle/working/edgeloop")

features = pd.read_csv(base / "stream_features_with_drift.csv")
predictions = pd.read_csv(base / "edge_predictions.csv")
manifest = pd.read_csv(base / "corruption_manifest.csv")

data = features[
    ["sample_id", "condition", "drift_flag"]
].merge(
    predictions[
        ["sample_id", "condition", "entropy", "predicted_class"]
    ],
    on=["sample_id", "condition"],
    validate="one_to_one"
).merge(
    manifest[["sample_id", "condition", "label"]],
    on=["sample_id", "condition"],
    validate="one_to_one"
)

data["is_error"] = (
    data["predicted_class"].str.lower()
    != data["label"].str.lower()
)

budget = 203

# Reserve a balanced portion of the budget for each condition.
# Within each condition, prioritize uncertain records.
conditions = sorted(data["condition"].unique())
per_condition = budget // len(conditions)
remainder = budget % len(conditions)

selected_parts = []

for i, condition in enumerate(conditions):
    group = data[data["condition"] == condition].sort_values(
        "entropy", ascending=False
    )
    quota = per_condition + (1 if i < remainder else 0)
    selected_parts.append(group.head(quota))

selected = pd.concat(selected_parts)
selected_ids = set(selected.index)

data["balanced_selected"] = data.index.isin(selected_ids)

print("Selected records:", int(data["balanced_selected"].sum()))
print("Unique source images:", data.loc[
    data["balanced_selected"], "sample_id"
].nunique())
print("Errors captured:", int(data.loc[
    data["balanced_selected"], "is_error"
].sum()))
print("Selected error rate:", round(
    data.loc[data["balanced_selected"], "is_error"].mean(), 4
))

print("\nResults by condition:")
print(
    data[data["balanced_selected"]]
    .groupby("condition")["is_error"]
    .agg(["count", "sum", "mean"])
    .round(4)
    .to_string()
)

data.to_csv(base / "balanced_selection_experiment.csv", index=False)
print("\nSaved balanced_selection_experiment.csv")

Selected records: 203
Unique source images: 126
Errors captured: 115
Selected error rate: 0.5665

Results by condition:
           count  sum    mean
condition                    
blur          51   26  0.5098
clean         51   27  0.5294
low_light     51   30  0.5882
noise         50   32  0.6400

Saved balanced_selection_experiment.csv


In [23]:
from pathlib import Path
import pandas as pd

base = Path("/kaggle/working/edgeloop")

manifest = pd.read_csv(base / "corruption_manifest.csv")
predictions = pd.read_csv(base / "edge_predictions.csv")
original = pd.read_csv(base / "selection_results.csv")
unique_aware = pd.read_csv(base / "selection_results_v2.csv")
balanced = pd.read_csv(base / "balanced_selection_experiment.csv")

data = predictions.merge(
    manifest[["sample_id", "condition", "label"]],
    on=["sample_id", "condition"],
    validate="one_to_one"
)
data["is_error"] = (
    data["predicted_class"].str.lower()
    != data["label"].str.lower()
)

# Attach strategy selections by record identity
keys = ["sample_id", "condition"]

data = data.merge(
    original[keys + ["random_selected", "uncertainty_selected"]],
    on=keys, validate="one_to_one"
)
data = data.merge(
    unique_aware[keys + ["unique_aware_selected"]],
    on=keys, validate="one_to_one"
)
data = data.merge(
    balanced[keys + ["balanced_selected"]],
    on=keys, validate="one_to_one"
)

strategies = [
    "random_selected",
    "uncertainty_selected",
    "unique_aware_selected",
    "balanced_selected",
]

rows = []
for strategy in strategies:
    selected = data[data[strategy].astype(bool)]
    rows.append({
        "strategy": strategy,
        "records": len(selected),
        "unique_images": selected["sample_id"].nunique(),
        "errors": int(selected["is_error"].sum()),
        "error_rate": round(selected["is_error"].mean(), 4),
        "unique_error_images": selected.loc[
            selected["is_error"], "sample_id"
        ].nunique()
    })

result = pd.DataFrame(rows)
print(result.to_string(index=False))
result.to_csv(base / "final_selection_comparison.csv", index=False)
print("\nSaved final_selection_comparison.csv")

             strategy  records  unique_images  errors  error_rate  unique_error_images
      random_selected      203            173      46      0.2266                   43
 uncertainty_selected      203            134     120      0.5911                   89
unique_aware_selected      203            203     102      0.5025                  102
    balanced_selected      203            126     115      0.5665                   86

Saved final_selection_comparison.csv


In [24]:
from pathlib import Path
import pandas as pd

base = Path("/kaggle/working/edgeloop")

required_files = [
    "dataset_manifest.csv",
    "selection_results.csv",
    "selection_results_v2.csv",
    "mobilenetv2_baseline.pth",
    "baseline_holdout_metrics.json",
]

print("CHECKPOINT FILES")
for name in required_files:
    path = base / name
    print(f"{name}: {'OK' if path.is_file() else 'MISSING'}")

manifest = pd.read_csv(base / "dataset_manifest.csv")
uncertainty = pd.read_csv(base / "selection_results.csv")
unique_aware = pd.read_csv(base / "selection_results_v2.csv")

print("\nDATASET MANIFEST COLUMNS")
print(manifest.columns.tolist())

print("\nDATASET SPLIT COUNTS")
print(manifest["split"].value_counts().to_string())

print("\nUNCERTAINTY SELECTION")
print("Records:", int(uncertainty["uncertainty_selected"].sum()))
print("Unique source images:",
      uncertainty.loc[
          uncertainty["uncertainty_selected"], "sample_id"
      ].nunique())

print("\nUNIQUE-IMAGE-AWARE SELECTION")
print("Records:", int(unique_aware["unique_aware_selected"].sum()))
print("Unique source images:",
      unique_aware.loc[
          unique_aware["unique_aware_selected"], "sample_id"
      ].nunique())

CHECKPOINT FILES
dataset_manifest.csv: OK
selection_results.csv: OK
selection_results_v2.csv: OK
mobilenetv2_baseline.pth: OK
baseline_holdout_metrics.json: OK

DATASET MANIFEST COLUMNS
['sample_id', 'path', 'label', 'split']

DATASET SPLIT COUNTS
split
train            1263
clean_holdout     506
edge_stream       506
validation        252

UNCERTAINTY SELECTION
Records: 203
Unique source images: 134

UNIQUE-IMAGE-AWARE SELECTION
Records: 203
Unique source images: 203


In [25]:
from pathlib import Path
import pandas as pd

base = Path("/kaggle/working/edgeloop")

# Load experiment files
manifest = pd.read_csv(base / "dataset_manifest.csv")
uncertainty = pd.read_csv(base / "selection_results.csv")
unique_aware = pd.read_csv(base / "selection_results_v2.csv")

# Map each original image to its dataset split
split_map = manifest.set_index("sample_id")["split"].to_dict()

# Selected records for each strategy
uncertain = uncertainty[
    uncertainty["uncertainty_selected"].astype(bool)
].copy()

diverse = unique_aware[
    unique_aware["unique_aware_selected"].astype(bool)
].copy()

# Identify selected images belonging to the training split only
uncertain["split"] = uncertain["sample_id"].map(split_map)
diverse["split"] = diverse["sample_id"].map(split_map)

# Exclude anything outside the original training split
uncertain_train = uncertain[uncertain["split"] == "train"].copy()
diverse_train = diverse[diverse["split"] == "train"].copy()

# Save safe candidate-training selections
uncertain_train.to_csv(
    base / "retraining_uncertainty_selection.csv", index=False
)
diverse_train.to_csv(
    base / "retraining_unique_aware_selection.csv", index=False
)

# Verify source paths exist
for name, frame in [
    ("uncertainty", uncertain_train),
    ("unique-aware", diverse_train),
]:
    valid_paths = frame["path"].map(lambda p: Path(p).is_file())

    print(
        f"{name}: selected={len(frame)}, "
        f"unique_images={frame['sample_id'].nunique()}, "
        f"train_split={len(frame)}, "
        f"existing_paths={int(valid_paths.sum())}, "
        f"missing_paths={int((~valid_paths).sum())}"
    )

print("\nPrepared candidate-training selection files.")
print("No model training has been run yet.")
print("Original validation and clean holdout remain excluded.")

uncertainty: selected=0, unique_images=0, train_split=0, existing_paths=0, missing_paths=0
unique-aware: selected=0, unique_images=0, train_split=0, existing_paths=0, missing_paths=0

Prepared candidate-training selection files.
No model training has been run yet.
Original validation and clean holdout remain excluded.


In [26]:
from pathlib import Path
import pandas as pd

base = Path("/kaggle/working/edgeloop")

manifest = pd.read_csv(base / "dataset_manifest.csv")
selection = pd.read_csv(base / "selection_results.csv")
unique = pd.read_csv(base / "selection_results_v2.csv")

def normalize_path(p):
    return str(p).replace("\\", "/").rstrip("/").lower()

def filename(p):
    return Path(str(p)).name.lower()

manifest["path_norm"] = manifest["path"].map(normalize_path)
manifest["filename_norm"] = manifest["path"].map(filename)

# Map each source image to its split and canonical path
by_id = manifest.drop_duplicates("sample_id").set_index("sample_id")
by_path = manifest.drop_duplicates("path_norm").set_index("path_norm")
by_filename = manifest.drop_duplicates("filename_norm").set_index("filename_norm")

def map_selection(frame, flag_column):
    chosen = frame[frame[flag_column].astype(bool)].copy()
    chosen["mapped_split"] = chosen["sample_id"].map(
        manifest.drop_duplicates("sample_id").set_index("sample_id")["split"]
    )

    # Fallback 1: match the original path
    missing = chosen["mapped_split"].isna()
    path_map = by_path["split"].to_dict()
    chosen.loc[missing, "mapped_split"] = (
        chosen.loc[missing, "path"].map(normalize_path).map(path_map)
    )

    # Fallback 2: match filename only when unique in the manifest
    missing = chosen["mapped_split"].isna()
    filename_map = by_filename["split"].to_dict()
    chosen.loc[missing, "mapped_split"] = (
        chosen.loc[missing, "path"].map(filename).map(filename_map)
    )

    print(f"\n{flag_column}")
    print("Selected records:", len(chosen))
    print("Mapped to train:", int((chosen["mapped_split"] == "train").sum()))
    print("Mapped to validation:", int((chosen["mapped_split"] == "validation").sum()))
    print("Mapped to clean holdout:", int((chosen["mapped_split"] == "clean_holdout").sum()))
    print("Mapped to edge stream:", int((chosen["mapped_split"] == "edge_stream").sum()))
    print("Unmapped:", int(chosen["mapped_split"].isna().sum()))

    return chosen

uncertain = map_selection(selection, "uncertainty_selected")
diverse = map_selection(unique, "unique_aware_selected")

print("\nManifest example:")
print(manifest[["sample_id", "path", "split"]].head(3).to_string(index=False))

print("\nSelection example:")
print(selection[["sample_id", "path", "condition"]].head(3).to_string(index=False))


uncertainty_selected
Selected records: 203
Mapped to train: 0
Mapped to validation: 0
Mapped to clean holdout: 0
Mapped to edge stream: 203
Unmapped: 0

unique_aware_selected
Selected records: 203
Mapped to train: 0
Mapped to validation: 0
Mapped to clean holdout: 0
Mapped to edge stream: 203
Unmapped: 0

Manifest example:
         sample_id                                                                                path split
    metal_metal124     /kaggle/input/datasets/balajimali/ml-project/dataset-resized/metal/metal124.jpg train
plastic_plastic279 /kaggle/input/datasets/balajimali/ml-project/dataset-resized/plastic/plastic279.jpg train
    glass_glass334     /kaggle/input/datasets/balajimali/ml-project/dataset-resized/glass/glass334.jpg train

Selection example:
     sample_id                                                                            path condition
glass_glass320 /kaggle/input/datasets/balajimali/ml-project/dataset-resized/glass/glass320.jpg     clean
glass_gl

In [27]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader

base = Path("/kaggle/working/edgeloop")
manifest = pd.read_csv(base / "dataset_manifest.csv")

# Only use the original training split to build training selections.
train_df = manifest[manifest["split"] == "train"].copy().reset_index(drop=True)
assert len(train_df) > 0, "Training split is empty."

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recreate the six-class MobileNetV2 architecture.
classes = sorted(manifest["label"].unique())
model = models.mobilenet_v2(weights=None)
model.classifier[1] = nn.Linear(model.last_channel, len(classes))

checkpoint = torch.load(
    base / "mobilenetv2_baseline.pth",
    map_location=device,
    weights_only=False
)

# Handle checkpoints saved either as a state dict or wrapped dictionary.
if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
elif isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

state_dict = {
    k.removeprefix("module."): v
    for k, v in state_dict.items()
}

model.load_state_dict(state_dict)
model.to(device).eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class ImageDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(row["path"]) as im:
            image = transform(im.convert("RGB"))
        return image, idx

loader = DataLoader(
    ImageDataset(train_df),
    batch_size=32,
    shuffle=False,
    num_workers=2
)

entropies = np.zeros(len(train_df), dtype=np.float32)
confidences = np.zeros(len(train_df), dtype=np.float32)

with torch.inference_mode():
    for images, indices in loader:
        logits = model(images.to(device))
        probs = torch.softmax(logits, dim=1)
        entropy = -(probs * probs.clamp_min(1e-12).log()).sum(dim=1)

        entropies[indices.numpy()] = entropy.cpu().numpy()
        confidences[indices.numpy()] = probs.max(dim=1).values.cpu().numpy()

train_df["entropy"] = entropies
train_df["confidence"] = confidences

budget = min(203, len(train_df))

# Strategy A: highest-uncertainty training images.
uncertainty_train = train_df.nlargest(budget, "entropy").copy()

# Strategy B: diverse unique images.
# Each original training image occurs once in this manifest, so the selected
# rows are unique source images. Select a range of uncertainty levels.
ordered = train_df.sort_values("entropy", ascending=False).reset_index(drop=True)
positions = np.linspace(0, len(ordered) - 1, budget, dtype=int)
unique_aware_train = ordered.iloc[positions].copy()

uncertainty_train.to_csv(
    base / "retraining_uncertainty_selection.csv", index=False
)
unique_aware_train.to_csv(
    base / "retraining_unique_aware_selection.csv", index=False
)

print("Training-only selection files created.")
print("Uncertainty strategy:", len(uncertainty_train),
      "unique images:", uncertainty_train["sample_id"].nunique())
print("Unique-aware strategy:", len(unique_aware_train),
      "unique images:", unique_aware_train["sample_id"].nunique())
print("All selected rows from train split:",
      (uncertainty_train["split"].eq("train").all()
       and unique_aware_train["split"].eq("train").all()))
print("Saved both selections in:", base)

Training-only selection files created.
Uncertainty strategy: 203 unique images: 203
Unique-aware strategy: 203 unique images: 203
All selected rows from train split: True
Saved both selections in: /kaggle/working/edgeloop


In [28]:
from pathlib import Path
import copy
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import models, transforms

base = Path("/kaggle/working/edgeloop")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

manifest = pd.read_csv(base / "dataset_manifest.csv")
classes = sorted(manifest["label"].unique())
class_to_idx = {name: i for i, name in enumerate(classes)}

train_df = manifest[manifest["split"] == "train"].copy()
val_df = manifest[manifest["split"] == "validation"].copy()

uncertain = pd.read_csv(base / "retraining_uncertainty_selection.csv")
diverse = pd.read_csv(base / "retraining_unique_aware_selection.csv")

# Confirm every selected example is from the original training split.
assert uncertain["split"].eq("train").all()
assert diverse["split"].eq("train").all()
assert set(uncertain["sample_id"]).issubset(set(train_df["sample_id"]))
assert set(diverse["sample_id"]).issubset(set(train_df["sample_id"]))

# Baseline preprocessing, matching the original evaluation pipeline.
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class WasteDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        row = self.frame.iloc[i]
        with Image.open(row["path"]) as im:
            x = image_transform(im.convert("RGB"))
        y = class_to_idx[row["label"]]
        return x, y

def make_model():
    model = models.mobilenet_v2(weights=None)
    model.classifier[1] = nn.Linear(model.last_channel, len(classes))

    checkpoint = torch.load(
        base / "mobilenetv2_baseline.pth",
        map_location="cpu",
        weights_only=False
    )
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        checkpoint = checkpoint["state_dict"]
    elif isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        checkpoint = checkpoint["model_state_dict"]

    checkpoint = {k.removeprefix("module."): v
                  for k, v in checkpoint.items()}
    model.load_state_dict(checkpoint)
    return model.to(device)

def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.inference_mode():
        for x, y in loader:
            logits = model(x.to(device))
            y_pred.extend(logits.argmax(1).cpu().tolist())
            y_true.extend(y.tolist())

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ))
    }

val_loader = DataLoader(
    WasteDataset(val_df), batch_size=32, shuffle=False, num_workers=2
)

# Controlled fine-tuning: identical settings and fixed validation split.
# Selected training images are already part of train_df, so do not duplicate
# them; instead, increase their sampling probability.
def train_candidate(name, selected_df, epochs=3):
    selected_ids = set(selected_df["sample_id"])
    weights = [
        2.0 if sid in selected_ids else 1.0
        for sid in train_df["sample_id"]
    ]

    generator = torch.Generator().manual_seed(seed)
    sampler = torch.utils.data.WeightedRandomSampler(
        weights, num_samples=len(train_df), replacement=True,
        generator=generator
    )

    train_loader = DataLoader(
        WasteDataset(train_df),
        batch_size=32,
        sampler=sampler,
        num_workers=2
    )

    model = make_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
    criterion = nn.CrossEntropyLoss()

    best_f1 = -1.0
    best_state = None
    history = []

    for epoch in range(epochs):
        model.train()
        total_loss, seen = 0.0, 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)
            seen += len(y)

        metrics = evaluate(model, val_loader)
        history.append({
            "epoch": epoch + 1,
            "train_loss": total_loss / seen,
            **metrics
        })

        if metrics["macro_f1"] > best_f1:
            best_f1 = metrics["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"{name} | epoch {epoch + 1}/{epochs} | "
            f"loss={total_loss / seen:.4f} | "
            f"val_acc={metrics['accuracy']:.4f} | "
            f"val_macro_f1={metrics['macro_f1']:.4f}"
        )

    model.load_state_dict(best_state)
    torch.save(best_state, base / f"{name}_candidate.pth")
    pd.DataFrame(history).to_csv(
        base / f"{name}_training_history.csv", index=False
    )

    best_metrics = max(history, key=lambda x: x["macro_f1"])
    with open(base / f"{name}_validation_metrics.json", "w") as f:
        json.dump(best_metrics, f, indent=2)

    return {
        "candidate": name,
        "best_epoch": best_metrics["epoch"],
        "val_accuracy": best_metrics["accuracy"],
        "val_macro_f1": best_metrics["macro_f1"]
    }

results = []
results.append(train_candidate("uncertainty", uncertain))
results.append(train_candidate("unique_aware", diverse))

comparison = pd.DataFrame(results)
comparison.to_csv(base / "retraining_candidate_comparison.csv", index=False)

print("\nVALIDATION COMPARISON")
print(comparison.to_string(index=False))
print("\nSaved candidate checkpoints and metrics to:", base)
print("Clean holdout was not used.")

uncertainty | epoch 1/3 | loss=0.0855 | val_acc=0.9206 | val_macro_f1=0.9053
uncertainty | epoch 2/3 | loss=0.0497 | val_acc=0.9206 | val_macro_f1=0.9007
uncertainty | epoch 3/3 | loss=0.0553 | val_acc=0.9206 | val_macro_f1=0.8968
unique_aware | epoch 1/3 | loss=0.0499 | val_acc=0.9206 | val_macro_f1=0.9006
unique_aware | epoch 2/3 | loss=0.0439 | val_acc=0.9167 | val_macro_f1=0.9006
unique_aware | epoch 3/3 | loss=0.0308 | val_acc=0.9127 | val_macro_f1=0.8927

VALIDATION COMPARISON
   candidate  best_epoch  val_accuracy  val_macro_f1
 uncertainty           1      0.920635      0.905302
unique_aware           1      0.920635      0.900565

Saved candidate checkpoints and metrics to: /kaggle/working/edgeloop
Clean holdout was not used.


In [29]:
from pathlib import Path
import json
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, f1_score, classification_report

base = Path("/kaggle/working/edgeloop")
manifest = pd.read_csv(base / "dataset_manifest.csv")
holdout = manifest[manifest["split"] == "clean_holdout"].copy()

classes = sorted(manifest["label"].unique())
class_to_idx = {name: i for i, name in enumerate(classes)}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class HoldoutDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        row = self.frame.iloc[i]
        with Image.open(row["path"]) as im:
            x = transform(im.convert("RGB"))
        return x, class_to_idx[row["label"]]

loader = DataLoader(
    HoldoutDataset(holdout),
    batch_size=32,
    shuffle=False,
    num_workers=2
)

def load_state(path):
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        checkpoint = checkpoint["state_dict"]
    elif isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        checkpoint = checkpoint["model_state_dict"]
    return {k.removeprefix("module."): v for k, v in checkpoint.items()}

def evaluate_checkpoint(name, checkpoint_path):
    model = models.mobilenet_v2(weights=None)
    model.classifier[1] = nn.Linear(model.last_channel, len(classes))
    model.load_state_dict(load_state(checkpoint_path))
    model.to(device).eval()

    y_true, y_pred = [], []
    with torch.inference_mode():
        for x, y in loader:
            predictions = model(x.to(device)).argmax(1).cpu().tolist()
            y_pred.extend(predictions)
            y_true.extend(y.tolist())

    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "classification_report": classification_report(
            y_true, y_pred, labels=list(range(len(classes))),
            target_names=classes, output_dict=True, zero_division=0
        )
    }

checkpoints = [
    ("baseline", base / "mobilenetv2_baseline.pth"),
    ("uncertainty_candidate", base / "uncertainty_candidate.pth"),
    ("unique_aware_candidate", base / "unique_aware_candidate.pth")
]

results = [
    evaluate_checkpoint(name, path)
    for name, path in checkpoints
]

summary = pd.DataFrame([
    {
        "model": r["model"],
        "holdout_accuracy": r["accuracy"],
        "holdout_macro_f1": r["macro_f1"]
    }
    for r in results
])

summary.to_csv(base / "retraining_holdout_comparison.csv", index=False)

with open(base / "retraining_holdout_detailed.json", "w") as f:
    json.dump(results, f, indent=2)

print("FINAL HOLDOUT COMPARISON")
print(summary.round(4).to_string(index=False))
print("\nSaved final evaluation files.")
print("Holdout images:", len(holdout))

FINAL HOLDOUT COMPARISON
                 model  holdout_accuracy  holdout_macro_f1
              baseline            0.9130            0.8943
 uncertainty_candidate            0.9051            0.8855
unique_aware_candidate            0.9032            0.8805

Saved final evaluation files.
Holdout images: 506


In [30]:
from pathlib import Path
import json
import shutil
import pandas as pd

base = Path("/kaggle/working/edgeloop")

# Record the final holdout outcome
comparison_path = base / "retraining_holdout_comparison.csv"
comparison = pd.read_csv(comparison_path)

baseline = comparison.loc[comparison["model"] == "baseline"].iloc[0]

checkpoint_path = base / "checkpoint_v0.3.json"
with open(checkpoint_path) as f:
    checkpoint = json.load(f)

checkpoint["retraining_experiment"] = {
    "status": "completed_initial_comparison",
    "evaluation": "fixed_clean_holdout",
    "baseline_accuracy": float(baseline["holdout_accuracy"]),
    "baseline_macro_f1": float(baseline["holdout_macro_f1"]),
    "candidate_results": comparison.to_dict(orient="records"),
    "promotion_decision": "keep_baseline",
    "reason": "Both retraining candidates scored below baseline on clean holdout."
}

with open(base / "checkpoint_v0.4.json", "w") as f:
    json.dump(checkpoint, f, indent=2)

# Export all current research artifacts
archive_path = shutil.make_archive(
    "/kaggle/working/EdgeLoop_Kaggle_Research_v0.5",
    "zip",
    root_dir="/kaggle/working",
    base_dir="edgeloop"
)

print("Saved checkpoint:", base / "checkpoint_v0.4.json")
print("Saved archive:", archive_path)
print("Archive size (MB):",
      round(Path(archive_path).stat().st_size / (1024 * 1024), 2))
print("\nPromotion decision: KEEP BASELINE")

Saved checkpoint: /kaggle/working/edgeloop/checkpoint_v0.4.json
Saved archive: /kaggle/working/EdgeLoop_Kaggle_Research_v0.5.zip
Archive size (MB): 115.47

Promotion decision: KEEP BASELINE


In [31]:
from pathlib import Path
import json
import pandas as pd

base = Path("/kaggle/working/edgeloop")

# 1. Compare final holdout results
holdout_path = base / "retraining_holdout_comparison.csv"
holdout = pd.read_csv(holdout_path)

print("=" * 65)
print("HOLDOUT PERFORMANCE COMPARISON")
print("=" * 65)
print(holdout.round(4).to_string(index=False))

# 2. Compare validation results
val_path = base / "retraining_candidate_comparison.csv"
if val_path.exists():
    print("\n" + "=" * 65)
    print("VALIDATION PERFORMANCE")
    print("=" * 65)
    print(pd.read_csv(val_path).round(4).to_string(index=False))

# 3. Review training curves
for name in ["uncertainty", "unique_aware"]:
    history_path = base / f"{name}_training_history.csv"
    if history_path.exists():
        history = pd.read_csv(history_path)
        print(f"\n{name.upper()} TRAINING HISTORY")
        print(history.round(4).to_string(index=False))

# 4. Per-class holdout F1 compared with the baseline
details_path = base / "retraining_holdout_detailed.json"
if details_path.exists():
    with open(details_path) as f:
        details = json.load(f)

    rows = []
    for result in details:
        report = result["classification_report"]
        for class_name, values in report.items():
            if isinstance(values, dict) and "f1-score" in values:
                rows.append({
                    "model": result["model"],
                    "class": class_name,
                    "precision": values["precision"],
                    "recall": values["recall"],
                    "f1": values["f1-score"],
                    "support": values["support"]
                })

    per_class = pd.DataFrame(rows)
    if not per_class.empty:
        pivot = per_class.pivot(
            index="class", columns="model", values="f1"
        )
        print("\n" + "=" * 65)
        print("PER-CLASS HOLDOUT F1")
        print("=" * 65)
        print(pivot.round(4).to_string())

        if {"baseline", "uncertainty_candidate",
            "unique_aware_candidate"}.issubset(pivot.columns):
            pivot["uncertainty_delta"] = (
                pivot["uncertainty_candidate"] - pivot["baseline"]
            )
            pivot["unique_aware_delta"] = (
                pivot["unique_aware_candidate"] - pivot["baseline"]
            )
            print("\nF1 CHANGE VS BASELINE")
            print(pivot[[
                "uncertainty_delta", "unique_aware_delta"
            ]].round(4).to_string())

print("\nDiagnostic complete. No models were modified or retrained.")

HOLDOUT PERFORMANCE COMPARISON
                 model  holdout_accuracy  holdout_macro_f1
              baseline            0.9130            0.8943
 uncertainty_candidate            0.9051            0.8855
unique_aware_candidate            0.9032            0.8805

VALIDATION PERFORMANCE
   candidate  best_epoch  val_accuracy  val_macro_f1
 uncertainty           1        0.9206        0.9053
unique_aware           1        0.9206        0.9006

UNCERTAINTY TRAINING HISTORY
 epoch  train_loss  accuracy  macro_f1
     1      0.0855    0.9206    0.9053
     2      0.0497    0.9206    0.9007
     3      0.0553    0.9206    0.8968

UNIQUE_AWARE TRAINING HISTORY
 epoch  train_loss  accuracy  macro_f1
     1      0.0499    0.9206    0.9006
     2      0.0439    0.9167    0.9006
     3      0.0308    0.9127    0.8927

PER-CLASS HOLDOUT F1
model         baseline  uncertainty_candidate  unique_aware_candidate
class                                                                
cardboard      

In [34]:

from pathlib import Path
import pandas as pd

ROOT = Path("/kaggle/working/edgeloop")
if not ROOT.exists():
    ROOT = Path("/kaggle/working")

files = [
    "final_selection_comparison.csv",
    "selection_results.csv",
    "selection_results_v2.csv",
    "balanced_selection_experiment.csv",
]

print("=" * 65)
print("EDGELOOP SELECTION STRATEGY COMPARISON")
print("=" * 65)

for filename in files:
    path = ROOT / filename

    if path.exists():
        df = pd.read_csv(path)
        print(f"\n--- {filename} ---")
        print(df.to_string(index=False))
    else:
        print(f"\nNot found: {filename}")

print("\nAnalysis complete. No training or checkpoint changes made.")


EDGELOOP SELECTION STRATEGY COMPARISON

--- final_selection_comparison.csv ---
             strategy  records  unique_images  errors  error_rate  unique_error_images
      random_selected      203            173      46      0.2266                   43
 uncertainty_selected      203            134     120      0.5911                   89
unique_aware_selected      203            203     102      0.5025                  102
    balanced_selected      203            126     115      0.5665                   86

--- selection_results.csv ---
             sample_id                                                                                    path condition predicted_class  confidence  entropy  random_selected  uncertainty_selected upload_reason
        glass_glass320         /kaggle/input/datasets/balajimali/ml-project/dataset-resized/glass/glass320.jpg     clean           glass    0.998179 0.015024            False                 False  not_selected
        glass_glass320           

In [35]:

from pathlib import Path
import pandas as pd

ROOT = Path("/kaggle/working/edgeloop")
if not ROOT.exists():
    ROOT = Path("/kaggle/working")

source = ROOT / "final_selection_comparison.csv"

if not source.exists():
    raise FileNotFoundError(
        f"Cannot find {source}. Run the previous cell and check the printed file paths."
    )

df = pd.read_csv(source)

print("=" * 72)
print("EDGELOOP — SELECTION STRATEGY RESULTS")
print("=" * 72)
print(df.to_string(index=False))

# Save a copy for the research report.
out = ROOT / "selection_strategy_research_summary.csv"
df.to_csv(out, index=False)

print(f"\nSaved research summary: {out}")

# Print the key result using the actual available column names.
print("\nAVAILABLE METRICS")
print(df.columns.tolist())

print("\nRESEARCH INTERPRETATION")
print(
    "Compare strategies at the same upload budget. Consider both errors captured "
    "and unique source images covered. The stream includes correlated variants, "
    "so selected records must not be treated as independent physical images."
)
print("\nNo models trained; no checkpoints changed; no holdout evaluation performed.")


EDGELOOP — SELECTION STRATEGY RESULTS
             strategy  records  unique_images  errors  error_rate  unique_error_images
      random_selected      203            173      46      0.2266                   43
 uncertainty_selected      203            134     120      0.5911                   89
unique_aware_selected      203            203     102      0.5025                  102
    balanced_selected      203            126     115      0.5665                   86

Saved research summary: /kaggle/working/edgeloop/selection_strategy_research_summary.csv

AVAILABLE METRICS
['strategy', 'records', 'unique_images', 'errors', 'error_rate', 'unique_error_images']

RESEARCH INTERPRETATION
Compare strategies at the same upload budget. Consider both errors captured and unique source images covered. The stream includes correlated variants, so selected records must not be treated as independent physical images.

No models trained; no checkpoints changed; no holdout evaluation performed.


In [36]:

from pathlib import Path
import pandas as pd

ROOT = Path("/kaggle/working/edgeloop")
if not ROOT.exists():
    ROOT = Path("/kaggle/working")

pred_path = ROOT / "edge_predictions.csv"
selection_path = ROOT / "selection_results.csv"

if not pred_path.exists():
    raise FileNotFoundError(f"Missing: {pred_path}")
if not selection_path.exists():
    raise FileNotFoundError(f"Missing: {selection_path}")

pred = pd.read_csv(pred_path)
sel = pd.read_csv(selection_path)

print("Prediction columns:", pred.columns.tolist())
print("Selection columns:", sel.columns.tolist())

# Discover the strategy and selection-indicator columns.
strategy_col = next(
    (c for c in ["strategy", "selection_strategy"] if c in sel.columns),
    None
)
selected_col = next(
    (c for c in ["selected", "is_selected"] if c in sel.columns),
    None
)

if strategy_col is None or selected_col is None:
    print("\nCould not identify strategy/selection columns automatically.")
    print("No files modified. Use the column lists above to adjust the cell.")
else:
    selected = sel[
        sel[selected_col].astype(str).str.lower().isin(["true", "1", "yes"])
    ].copy()

    # Join on a shared record identifier when available.
    keys = [c for c in ["sample_id", "path"] if c in pred.columns and c in selected.columns]
    if not keys:
        print("\nNo shared record key found; cannot safely join condition labels.")
    else:
        merged = selected.merge(
            pred[keys + [c for c in ["condition", "predicted_class"] if c in pred.columns]],
            on=keys,
            how="left",
            validate="many_to_one"
        )

        if "condition" not in merged.columns:
            print("\nCondition column unavailable after joining.")
        else:
            summary = (
                merged.groupby([strategy_col, "condition"], dropna=False)
                .size()
                .reset_index(name="selected_records")
            )

            out = ROOT / "selection_condition_coverage.csv"
            summary.to_csv(out, index=False)

            print("\nSELECTION COUNTS BY CONDITION")
            print(summary.to_string(index=False))
            print(f"\nSaved: {out}")
            print("\nRead-only analysis complete; no models or checkpoints changed.")


Prediction columns: ['sample_id', 'path', 'condition', 'predicted_class', 'confidence', 'entropy']
Selection columns: ['sample_id', 'path', 'condition', 'predicted_class', 'confidence', 'entropy', 'random_selected', 'uncertainty_selected', 'upload_reason']

Could not identify strategy/selection columns automatically.
No files modified. Use the column lists above to adjust the cell.


In [37]:

from pathlib import Path
import pandas as pd

ROOT = Path("/kaggle/working/edgeloop")
if not ROOT.exists():
    ROOT = Path("/kaggle/working")

path = ROOT / "selection_results.csv"
df = pd.read_csv(path)

strategies = ["random_selected", "uncertainty_selected"]

required = ["condition", "sample_id"] + strategies
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

rows = []

for strategy in strategies:
    selected = df[
        df[strategy].astype(str).str.lower().isin(["true", "1", "yes"])
    ].copy()

    for condition, group in selected.groupby("condition"):
        rows.append({
            "strategy": strategy.replace("_selected", ""),
            "condition": condition,
            "selected_records": len(group),
            "unique_source_images": group["sample_id"].nunique(),
            "mean_confidence": (
                round(group["confidence"].mean(), 4)
                if "confidence" in group.columns else None
            ),
            "mean_entropy": (
                round(group["entropy"].mean(), 4)
                if "entropy" in group.columns else None
            )
        })

summary = pd.DataFrame(rows)

out = ROOT / "selection_condition_coverage.csv"
summary.to_csv(out, index=False)

print("=" * 65)
print("SELECTION COVERAGE BY CONDITION")
print("=" * 65)
print(summary.to_string(index=False))

print("\nTOTALS BY STRATEGY")
print(
    summary.groupby("strategy")[
        ["selected_records", "unique_source_images"]
    ].sum().to_string()
)

print(f"\nSaved: {out}")
print("Read-only analysis: no model training or checkpoint changes.")


SELECTION COVERAGE BY CONDITION
   strategy condition  selected_records  unique_source_images  mean_confidence  mean_entropy
     random      blur                44                    44           0.8764        0.3445
     random     clean                55                    55           0.9270        0.2308
     random low_light                64                    64           0.8140        0.5185
     random     noise                40                    40           0.8137        0.5385
uncertainty      blur                42                    42           0.4594        1.2907
uncertainty     clean                12                    12           0.4618        1.2703
uncertainty low_light                54                    54           0.4921        1.2905
uncertainty     noise                95                    95           0.4721        1.3424

TOTALS BY STRATEGY
             selected_records  unique_source_images
strategy                                           
random 

In [38]:

from pathlib import Path
import pandas as pd

ROOT = Path("/kaggle/working/edgeloop")
if not ROOT.exists():
    ROOT = Path("/kaggle/working")

# Locate existing drift-aware experiment outputs.
candidates = [
    ROOT / "final_selection_comparison.csv",
    ROOT / "selection_results.csv",
    ROOT / "selection_results_v2.csv",
    ROOT / "balanced_selection_experiment.csv",
]

print("=" * 68)
print("EDGELOOP — DRIFT-AWARE STRATEGY AUDIT")
print("=" * 68)

for path in candidates:
    if path.exists():
        df = pd.read_csv(path)
        print(f"\n{path.name}")
        print("Columns:", df.columns.tolist())

        # Print only rows or columns that identify drift-aware results.
        if "strategy" in df.columns:
            mask = df["strategy"].astype(str).str.contains(
                "drift", case=False, na=False
            )
            if mask.any():
                print(df.loc[mask].to_string(index=False))

        for col in df.columns:
            if "drift" in col.lower() or "selected" in col.lower():
                print(f"Relevant column: {col}")

print("\nAudit only: no training, checkpoint changes, or holdout evaluation.")


EDGELOOP — DRIFT-AWARE STRATEGY AUDIT

final_selection_comparison.csv
Columns: ['strategy', 'records', 'unique_images', 'errors', 'error_rate', 'unique_error_images']

selection_results.csv
Columns: ['sample_id', 'path', 'condition', 'predicted_class', 'confidence', 'entropy', 'random_selected', 'uncertainty_selected', 'upload_reason']
Relevant column: random_selected
Relevant column: uncertainty_selected

selection_results_v2.csv
Columns: ['sample_id', 'path', 'condition', 'predicted_class', 'confidence', 'entropy', 'random_selected', 'uncertainty_selected', 'upload_reason', 'unique_aware_selected']
Relevant column: random_selected
Relevant column: uncertainty_selected
Relevant column: unique_aware_selected

balanced_selection_experiment.csv
Columns: ['sample_id', 'condition', 'drift_flag', 'entropy', 'predicted_class', 'label', 'is_error', 'balanced_selected']
Relevant column: drift_flag
Relevant column: balanced_selected

Audit only: no training, checkpoint changes, or holdout evalu

In [39]:

from pathlib import Path
import pandas as pd

ROOT = Path("/kaggle/working/edgeloop")
if not ROOT.exists():
    ROOT = Path("/kaggle/working")

required_files = [
    "edge_predictions.csv",
    "stream_features.csv",
    "selection_results.csv",
    "selection_results_v2.csv",
    "balanced_selection_experiment.csv",
    "final_selection_comparison.csv",
]

print("=" * 68)
print("EDGELOOP — DRIFT-AWARE EXPERIMENT READINESS")
print("=" * 68)

for filename in required_files:
    path = ROOT / filename
    if path.exists():
        df = pd.read_csv(path, nrows=5)
        print(f"\n{filename}: FOUND")
        print("Columns:", df.columns.tolist())
    else:
        print(f"\n{filename}: MISSING")

features_path = ROOT / "stream_features.csv"
if features_path.exists():
    features = pd.read_csv(features_path)
    print("\nDRIFT FEATURE COVERAGE")
    if "condition" in features.columns:
        print(features["condition"].value_counts().to_string())
    else:
        print("No condition column found.")

balanced_path = ROOT / "balanced_selection_experiment.csv"
if balanced_path.exists():
    balanced = pd.read_csv(balanced_path)
    if {"condition", "drift_flag", "balanced_selected"}.issubset(balanced.columns):
        print("\nSAVED BALANCED-SELECTION DRIFT FLAGS")
        print(
            balanced.groupby(["condition", "drift_flag"])["balanced_selected"]
            .sum()
            .rename("selected_records")
            .to_string()
        )

print("\nReadiness check complete. No training or checkpoint changes.")


EDGELOOP — DRIFT-AWARE EXPERIMENT READINESS

edge_predictions.csv: FOUND
Columns: ['sample_id', 'path', 'condition', 'predicted_class', 'confidence', 'entropy']

stream_features.csv: FOUND
Columns: ['sample_id', 'path', 'label', 'condition', 'high_frequency_std', 'high_frequency_mean_abs', 'brightness', 'contrast', 'edge_strength']

selection_results.csv: FOUND
Columns: ['sample_id', 'path', 'condition', 'predicted_class', 'confidence', 'entropy', 'random_selected', 'uncertainty_selected', 'upload_reason']

selection_results_v2.csv: FOUND
Columns: ['sample_id', 'path', 'condition', 'predicted_class', 'confidence', 'entropy', 'random_selected', 'uncertainty_selected', 'upload_reason', 'unique_aware_selected']

balanced_selection_experiment.csv: FOUND
Columns: ['sample_id', 'condition', 'drift_flag', 'entropy', 'predicted_class', 'label', 'is_error', 'balanced_selected']

final_selection_comparison.csv: FOUND
Columns: ['strategy', 'records', 'unique_images', 'errors', 'error_rate', 'uniq

In [40]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/kaggle/working/edgeloop")
if not ROOT.exists():
    ROOT = Path("/kaggle/working")

BUDGET = 203
SEED = 42

pred = pd.read_csv(ROOT / "edge_predictions.csv")
features = pd.read_csv(ROOT / "stream_features.csv")
existing = pd.read_csv(ROOT / "selection_results_v2.csv")

# Join predictions with features and ground-truth labels.
keys = ["sample_id", "path", "condition"]
df = pred.merge(
    features[keys + ["label", "brightness", "contrast",
                     "edge_strength", "high_frequency_mean_abs"]],
    on=keys,
    how="left",
    validate="one_to_one"
)

if df["label"].isna().any():
    raise ValueError("Some labels did not join. Check sample_id/path alignment.")

df["is_error"] = (
    df["predicted_class"].astype(str) != df["label"].astype(str)
)

# Reconstruct the existing clean-reference thresholds.
clean = df[df["condition"] == "clean"]
brightness_low = clean["brightness"].quantile(0.05)
brightness_high = clean["brightness"].quantile(0.95)
edge_min = clean["edge_strength"].quantile(0.05)
hf_min = clean["high_frequency_mean_abs"].quantile(0.05)
hf_max = clean["high_frequency_mean_abs"].quantile(0.95)

df["drift_flag"] = (
    (df["brightness"] < brightness_low) |
    (df["brightness"] > brightness_high) |
    (df["edge_strength"] < edge_min) |
    (df["high_frequency_mean_abs"] < hf_min) |
    (df["high_frequency_mean_abs"] > hf_max)
)

# Align existing selection flags using stable record keys.
flag_cols = [
    "random_selected",
    "uncertainty_selected",
    "unique_aware_selected"
]
available = [c for c in flag_cols if c in existing.columns]
if available:
    df = df.merge(
        existing[keys + available],
        on=keys,
        how="left",
        validate="one_to_one"
    )

def top_n(frame, n, sort_cols, ascending):
    return frame.sort_values(
        sort_cols, ascending=ascending, kind="mergesort"
    ).head(n).copy()

# Drift-first, entropy-second. Same total upload budget.
drift_selected = top_n(
    df, BUDGET, ["drift_flag", "entropy"], [False, False]
)
df["drift_aware_selected"] = df.index.isin(drift_selected.index)

# Produce metrics for existing strategies and reconstructed drift-aware.
strategies = {
    "drift_aware": "drift_aware_selected"
}
for col in available:
    strategies[col.replace("_selected", "")] = col

summary_rows = []
coverage_rows = []

for name, col in strategies.items():
    if name == "drift_aware":
        chosen = drift_selected
    else:
        chosen = df[df[col].fillna(False).astype(bool)]

    summary_rows.append({
        "strategy": name,
        "records": len(chosen),
        "unique_images": chosen["sample_id"].nunique(),
        "errors": int(chosen["is_error"].sum()),
        "error_rate": round(chosen["is_error"].mean(), 4),
        "unique_error_images": chosen.loc[
            chosen["is_error"], "sample_id"
        ].nunique()
    })

    for condition, group in chosen.groupby("condition"):
        coverage_rows.append({
            "strategy": name,
            "condition": condition,
            "selected_records": len(group),
            "errors": int(group["is_error"].sum()),
            "error_rate": round(group["is_error"].mean(), 4)
        })

summary = pd.DataFrame(summary_rows)
coverage = pd.DataFrame(coverage_rows)

summary.to_csv(ROOT / "drift_aware_selection_comparison.csv", index=False)
coverage.to_csv(ROOT / "drift_aware_condition_coverage.csv", index=False)
df.to_csv(ROOT / "drift_aware_reconstructed_records.csv", index=False)

print("=" * 68)
print("STRATEGY COMPARISON")
print(summary.to_string(index=False))
print("\nCONDITION COVERAGE AND ERROR CAPTURE")
print(coverage.to_string(index=False))
print(f"\nDrift flags: {int(df['drift_flag'].sum())}/{len(df)} records")
print(f"Saved outputs in: {ROOT}")
print("\nNo model training, checkpoint changes, or holdout evaluation.")


STRATEGY COMPARISON
    strategy  records  unique_images  errors  error_rate  unique_error_images
 drift_aware      203            163      94      0.4631                   84
      random      203            173      46      0.2266                   43
 uncertainty      203            134     120      0.5911                   89
unique_aware      203            203     102      0.5025                  102

CONDITION COVERAGE AND ERROR CAPTURE
    strategy condition  selected_records  errors  error_rate
 drift_aware      blur                26      11      0.4231
 drift_aware     clean                13       7      0.5385
 drift_aware low_light               138      62      0.4493
 drift_aware     noise                26      14      0.5385
      random      blur                44       7      0.1591
      random     clean                55       8      0.1455
      random low_light                64      19      0.2969
      random     noise                40      12      0.3000
 un

In [41]:

from pathlib import Path
import pandas as pd
import json

ROOT = Path("/kaggle/working/edgeloop")
if not ROOT.exists():
    ROOT = Path("/kaggle/working")

# Files and artifacts to verify
expected = [
    "mobilenetv2_baseline.pth",
    "baseline_holdout_metrics.json",
    "edge_predictions.csv",
    "corruption_manifest.csv",
    "stream_features.csv",
    "selection_results.csv",
    "selection_results_v2.csv",
    "balanced_selection_experiment.csv",
    "final_selection_comparison.csv",
    "selection_strategy_research_summary.csv",
    "selection_condition_coverage.csv",
    "drift_aware_selection_comparison.csv",
    "drift_aware_condition_coverage.csv",
    "drift_aware_reconstructed_records.csv",
    "retraining_holdout_comparison.csv",
    "retraining_holdout_detailed.json",
    "uncertainty_training_history.csv",
    "unique_aware_training_history.csv",
    "uncertainty_candidate.pth",
    "unique_aware_candidate.pth",
    "checkpoint_v0.3.json",
    "checkpoint_v0.4.json",
]

# Also search for the archive in the working directory.
archive_name = "EdgeLoop_Kaggle_Research_v0.5.zip"

print("=" * 76)
print("EDGELOOP — FINAL RESEARCH ARTIFACT AUDIT")
print("=" * 76)
print("Audit directory:", ROOT)

inventory = []
for name in expected + [archive_name]:
    p = ROOT / name
    exists = p.is_file()
    size_mb = round(p.stat().st_size / (1024 * 1024), 2) if exists else None

    inventory.append({
        "artifact": name,
        "status": "FOUND" if exists else "MISSING",
        "size_MB": size_mb
    })

inventory_df = pd.DataFrame(inventory)
print("\nARTIFACT INVENTORY")
print(inventory_df.to_string(index=False))

# Summarize important existing result files.
print("\n" + "=" * 76)
print("KEY RESULTS")
print("=" * 76)

for name in [
    "baseline_holdout_metrics.json",
    "retraining_holdout_comparison.csv",
    "final_selection_comparison.csv",
    "drift_aware_selection_comparison.csv",
]:
    p = ROOT / name
    if not p.exists():
        print(f"\n{name}: unavailable")
        continue

    print(f"\n--- {name} ---")
    try:
        if p.suffix == ".json":
            with open(p, "r") as f:
                obj = json.load(f)
            print(json.dumps(obj, indent=2)[:5000])
        else:
            print(pd.read_csv(p).to_string(index=False))
    except Exception as e:
        print("Could not read file:", str(e))

# Save the audit inventory and summary.
inventory_path = ROOT / "edgeloop_artifact_inventory.csv"
inventory_df.to_csv(inventory_path, index=False)

summary = {
    "audit_directory": str(ROOT),
    "artifacts_expected": len(expected) + 1,
    "artifacts_found": int((inventory_df["status"] == "FOUND").sum()),
    "artifacts_missing": int((inventory_df["status"] == "MISSING").sum()),
    "baseline_checkpoint_present": (ROOT / "mobilenetv2_baseline.pth").is_file(),
    "research_archive_present": (ROOT / archive_name).is_file(),
    "models_modified": False,
    "models_retrained": False,
    "holdout_evaluated": False
}

summary_path = ROOT / "edgeloop_artifact_audit.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 76)
print("AUDIT SUMMARY")
print("=" * 76)
print(json.dumps(summary, indent=2))
print("\nSaved:", inventory_path)
print("Saved:", summary_path)
print("\nAudit finished. No model training or checkpoint modifications.")


EDGELOOP — FINAL RESEARCH ARTIFACT AUDIT
Audit directory: /kaggle/working/edgeloop

ARTIFACT INVENTORY
                               artifact  status  size_MB
               mobilenetv2_baseline.pth   FOUND     8.75
          baseline_holdout_metrics.json   FOUND     0.00
                   edge_predictions.csv   FOUND     0.27
                corruption_manifest.csv   FOUND     0.20
                    stream_features.csv   FOUND     0.33
                  selection_results.csv   FOUND     0.32
               selection_results_v2.csv   FOUND     0.33
      balanced_selection_experiment.csv   FOUND     0.14
         final_selection_comparison.csv   FOUND     0.00
selection_strategy_research_summary.csv   FOUND     0.00
       selection_condition_coverage.csv   FOUND     0.00
   drift_aware_selection_comparison.csv   FOUND     0.00
     drift_aware_condition_coverage.csv   FOUND     0.00
  drift_aware_reconstructed_records.csv   FOUND     0.44
      retraining_holdout_comparison.csv   

In [42]:

from pathlib import Path
import zipfile

BASE = Path("/kaggle/working/edgeloop")
WORK = Path("/kaggle/working")
ARCHIVE_NAME = "EdgeLoop_Kaggle_Research_v0.5.zip"
archive = WORK / ARCHIVE_NAME

print("=" * 68)
print("EDGELOOP — RESEARCH ARCHIVE CHECK")
print("=" * 68)

# Search the working directory before rebuilding.
found = [
    p for p in WORK.rglob(ARCHIVE_NAME)
    if p.is_file() and p != archive
]

if archive.exists():
    print("Existing archive found:", archive)
    print("Size (MB):", round(archive.stat().st_size / (1024 * 1024), 2))

elif found:
    archive = found[0]
    print("Existing archive found:", archive)
    print("Size (MB):", round(archive.stat().st_size / (1024 * 1024), 2))

else:
    if not BASE.exists():
        raise FileNotFoundError(f"Research directory missing: {BASE}")

    # Include saved research outputs and model checkpoints, not raw images.
    included = []
    for p in BASE.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() in {".csv", ".json", ".pth"}:
            included.append(p)

    if not included:
        raise RuntimeError("No research artifacts found to archive.")

    with zipfile.ZipFile(
        archive, mode="w", compression=zipfile.ZIP_DEFLATED
    ) as zf:
        for p in sorted(included):
            zf.write(p, arcname=str(Path("edgeloop") / p.relative_to(BASE)))

    print("Archive rebuilt:", archive)
    print("Files included:", len(included))
    print("Size (MB):", round(archive.stat().st_size / (1024 * 1024), 2))

# Verify that the ZIP is readable.
with zipfile.ZipFile(archive, "r") as zf:
    bad_member = zf.testzip()
    if bad_member is not None:
        raise RuntimeError(f"Archive verification failed at: {bad_member}")
    print("ZIP verification: PASSED")
    print("Archived files:", len(zf.namelist()))

print("\nNo models retrained. No checkpoints modified.")
print("Download from Kaggle's /kaggle/working output files.")


EDGELOOP — RESEARCH ARCHIVE CHECK
Existing archive found: /kaggle/working/EdgeLoop_Kaggle_Research_v0.5.zip
Size (MB): 115.47
ZIP verification: PASSED
Archived files: 1554

No models retrained. No checkpoints modified.
Download from Kaggle's /kaggle/working output files.
